# Lab 2: Image Preprocessing & Augmentation Layers

In this lab, you will learn how to preprocess and augment images using **Keras 3 built-in layers** with the **PyTorch backend**.

## Why Preprocessing and Augmentation Matter

**Preprocessing** ensures your data is in a consistent format that neural networks can learn from efficiently. Raw pixel values (0-255) create large gradients that make training unstable -- rescaling to [0, 1] fixes this.

**Data augmentation** artificially expands your training set by applying random transformations (flips, rotations, zoom, brightness changes) to each image during training. This forces the model to learn features that are invariant to these transformations, which dramatically reduces overfitting -- especially when you have limited training data.

The key insight: by embedding these operations as Keras layers, they become part of the model itself. This means preprocessing is automatically applied during inference too, so you never have to worry about mismatched preprocessing between training and deployment.

In [ ]:
# Run this cell in Google Colab to install dependencies
# Skip if running locally with uv
import sys
if 'google.colab' in sys.modules:
    !pip install -q keras torch torchvision python-dotenv datasets transformers huggingface_hub
    print('Dependencies installed!')

In [ ]:
# Set the backend to PyTorch BEFORE importing Keras
import os
os.environ["KERAS_BACKEND"] = "torch"

import keras
import numpy as np
import matplotlib.pyplot as plt

print(f"Keras version: {keras.__version__}")
print(f"Backend: {keras.backend.backend()}")

## Step 1: Load and Explore CIFAR-10

CIFAR-10 is a dataset of 60,000 32x32 color images in 10 classes. Unlike MNIST's grayscale digits, CIFAR-10 contains real-world objects in color, making it a much harder classification task.

In [ ]:
# Load CIFAR-10
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

# Class names for CIFAR-10
class_names = [
    "airplane", "automobile", "bird", "cat", "deer",
    "dog", "frog", "horse", "ship", "truck"
]

print(f"Training images shape: {x_train.shape}")
print(f"Training labels shape: {y_train.shape}")
print(f"Test images shape:     {x_test.shape}")
print(f"Test labels shape:     {y_test.shape}")
print(f"Pixel value range:     [{x_train.min()}, {x_train.max()}]")
print(f"Number of classes:     {len(class_names)}")

# Visualize sample images
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i, ax in enumerate(axes.flat):
    ax.imshow(x_train[i])
    ax.set_title(class_names[y_train[i][0]])
    ax.axis("off")
plt.suptitle("Sample CIFAR-10 Images", fontsize=14)
plt.tight_layout()
plt.show()

## Step 2: Build a Preprocessing Pipeline

We will create a preprocessing pipeline using Keras layers:

- **Rescaling(1./255)**: Normalizes pixel values from [0, 255] to [0, 1].
- **Resizing(32, 32)**: Ensures consistent image dimensions (already 32x32 for CIFAR-10, but this makes the pipeline reusable for other datasets).

Let's apply it and verify the transformation visually.

In [ ]:
# Build preprocessing pipeline
preprocessing = keras.Sequential([
    keras.layers.Rescaling(1.0 / 255),
    keras.layers.Resizing(32, 32),
], name="preprocessing")

# Apply to a batch of sample images
sample_images = x_train[:5].astype("float32")
preprocessed_images = preprocessing(sample_images)

# Convert to numpy for visualization
preprocessed_np = np.array(preprocessed_images)

print(f"Before preprocessing - shape: {sample_images.shape}, range: [{sample_images.min():.1f}, {sample_images.max():.1f}]")
print(f"After preprocessing  - shape: {preprocessed_np.shape}, range: [{preprocessed_np.min():.3f}, {preprocessed_np.max():.3f}]")

# Show before and after
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
for i in range(5):
    # Before
    axes[0, i].imshow(sample_images[i].astype("uint8"))
    axes[0, i].set_title("Before")
    axes[0, i].axis("off")
    # After
    axes[1, i].imshow(preprocessed_np[i])
    axes[1, i].set_title("After")
    axes[1, i].axis("off")
plt.suptitle("Preprocessing: Before vs After", fontsize=14)
plt.tight_layout()
plt.show()

## Step 3: Build an Augmentation Pipeline

Data augmentation applies random transformations during training to create variety:

- **RandomFlip("horizontal")**: Mirrors images left-to-right.
- **RandomRotation(0.1)**: Rotates up to 10% of a full circle (36 degrees).
- **RandomZoom(0.1)**: Zooms in or out by up to 10%.
- **RandomBrightness(0.2)**: Adjusts brightness by up to 20%.
- **RandomContrast(0.2)**: Adjusts contrast by up to 20%.

Let's visualize what these transformations do to a single image.

In [ ]:
# Build augmentation pipeline
augmentation = keras.Sequential([
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomZoom(0.1),
    keras.layers.RandomBrightness(0.2),
    keras.layers.RandomContrast(0.2),
], name="augmentation")

# Pick one image and show 8 augmented versions
original_image = x_train[7:8].astype("float32") / 255.0  # Normalize first

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
for i, ax in enumerate(axes.flat):
    augmented = augmentation(original_image, training=True)
    augmented_np = np.array(augmented[0])
    # Clip values to valid range for display
    augmented_np = np.clip(augmented_np, 0, 1)
    ax.imshow(augmented_np)
    ax.set_title(f"Augmented #{i+1}")
    ax.axis("off")

plt.suptitle(f"8 Augmented Versions of One Image (class: {class_names[y_train[7][0]]})", fontsize=14)
plt.tight_layout()
plt.show()

## Step 4: Integrate into a CNN Model

The power of Keras augmentation layers is that they can be embedded directly inside the model. During training, augmentation is applied on-the-fly. During inference, these layers are automatically disabled (they become no-ops).

We will build a CNN with augmentation and preprocessing as the first layers.

In [ ]:
# Prepare data
x_train_float = x_train.astype("float32")
x_test_float = x_test.astype("float32")

# One-hot encode labels
num_classes = 10
y_train_cat = keras.utils.to_categorical(y_train, num_classes)
y_test_cat = keras.utils.to_categorical(y_test, num_classes)

# Build CNN with augmentation layers integrated
model_with_aug = keras.Sequential([
    # Input
    keras.layers.Input(shape=(32, 32, 3)),
    # Preprocessing
    keras.layers.Rescaling(1.0 / 255),
    # Augmentation (only active during training)
    keras.layers.RandomFlip("horizontal"),
    keras.layers.RandomRotation(0.1),
    keras.layers.RandomZoom(0.1),
    keras.layers.RandomBrightness(0.2),
    keras.layers.RandomContrast(0.2),
    # Feature extraction
    keras.layers.Conv2D(32, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation="relu"),
    # Classification
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(num_classes, activation="softmax"),
], name="cnn_with_augmentation")

model_with_aug.summary()

# Compile and train
model_with_aug.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

print("\nTraining CNN with augmentation...")
history_with_aug = model_with_aug.fit(
    x_train_float,
    y_train_cat,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
)

print("\nTraining complete!")

## Step 5: Compare With and Without Augmentation

To see the impact of augmentation, we train the exact same CNN architecture without the augmentation layers and compare the learning curves.

Key things to look for:
- **Without augmentation**: Training accuracy rises quickly, but validation accuracy plateaus or drops -- a sign of overfitting.
- **With augmentation**: Training accuracy rises more slowly, but validation accuracy is closer to training accuracy -- better generalization.

In [ ]:
# Build the same CNN WITHOUT augmentation
model_without_aug = keras.Sequential([
    # Input
    keras.layers.Input(shape=(32, 32, 3)),
    # Preprocessing only (no augmentation)
    keras.layers.Rescaling(1.0 / 255),
    # Feature extraction (same architecture)
    keras.layers.Conv2D(32, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation="relu"),
    keras.layers.MaxPooling2D((2, 2)),
    keras.layers.Conv2D(64, (3, 3), activation="relu"),
    # Classification
    keras.layers.Flatten(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.Dense(num_classes, activation="softmax"),
], name="cnn_without_augmentation")

model_without_aug.compile(
    optimizer="adam",
    loss="categorical_crossentropy",
    metrics=["accuracy"],
)

print("Training CNN without augmentation...")
history_without_aug = model_without_aug.fit(
    x_train_float,
    y_train_cat,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
)

print("\nTraining complete!")

# Compare learning curves side by side
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Accuracy comparison
axes[0].plot(history_with_aug.history["accuracy"], label="Train (with aug)", linestyle="-", color="blue")
axes[0].plot(history_with_aug.history["val_accuracy"], label="Val (with aug)", linestyle="--", color="blue")
axes[0].plot(history_without_aug.history["accuracy"], label="Train (no aug)", linestyle="-", color="red")
axes[0].plot(history_without_aug.history["val_accuracy"], label="Val (no aug)", linestyle="--", color="red")
axes[0].set_title("Accuracy: With vs Without Augmentation")
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("Accuracy")
axes[0].legend()
axes[0].grid(True)

# Loss comparison
axes[1].plot(history_with_aug.history["loss"], label="Train (with aug)", linestyle="-", color="blue")
axes[1].plot(history_with_aug.history["val_loss"], label="Val (with aug)", linestyle="--", color="blue")
axes[1].plot(history_without_aug.history["loss"], label="Train (no aug)", linestyle="-", color="red")
axes[1].plot(history_without_aug.history["val_loss"], label="Val (no aug)", linestyle="--", color="red")
axes[1].set_title("Loss: With vs Without Augmentation")
axes[1].set_xlabel("Epoch")
axes[1].set_ylabel("Loss")
axes[1].legend()
axes[1].grid(True)

plt.tight_layout()
plt.show()

# Print final test metrics for both models
test_loss_aug, test_acc_aug = model_with_aug.evaluate(x_test_float, y_test_cat, verbose=0)
test_loss_no_aug, test_acc_no_aug = model_without_aug.evaluate(x_test_float, y_test_cat, verbose=0)

print(f"\n{'Model':<30} {'Test Loss':<12} {'Test Accuracy'}")
print(f"{'-'*55}")
print(f"{'With Augmentation':<30} {test_loss_aug:<12.4f} {test_acc_aug:.4f}")
print(f"{'Without Augmentation':<30} {test_loss_no_aug:<12.4f} {test_acc_no_aug:.4f}")

## Vibe-Coding Exercise

Now it is your turn to practice vibe coding! Try giving Claude the following prompt:

> "Add RandAugment to my CIFAR-10 augmentation pipeline. RandAugment applies a random sequence of N transformations from a predefined set, each with magnitude M. Show me how to implement it with Keras 3 and compare the results with my current augmentation."

### Other ideas to explore:

- **"Add CutMix augmentation to the training loop."** -- CutMix cuts and pastes patches between training images and mixes their labels proportionally.
- **"Visualize each augmentation layer individually."** -- Apply each transformation one at a time to see its isolated effect.
- **"Try training for 30 epochs with a learning rate scheduler."** -- More epochs with augmentation often yields significantly better results.
- **"Add early stopping to prevent wasted training time."** -- Stop training automatically when validation loss stops improving.

Remember: the goal of vibe coding is to describe what you want clearly, let the AI generate the code, then review and iterate. You are the architect -- the AI is the builder.